In [ ]:
import os
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets, utils
from PIL import Image
import os
import itertools
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import torch.nn.functional as F
import gc
import multiprocessing
from torch.optim.lr_scheduler import StepLR

In [ ]:
root_dir = "data"

In [ ]:
image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.gif')

Triedy ktoré nemajú konzistentné rozlíšenie:cityscapes,iphone2dslr_flower,maps,

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Hyperparametre

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 8
LR = 0.0002
EPOCHS = 50
LAMBDA_CYCLE = 10.0
LAMBDA_IDENTITY = 5.0
DROPOUT = 0.5
path_to_dataset="data/horse2zebra/horse2zebra"

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, root, transform=None, mode='train', max_samples=None):
        self.transform = transform
        self.root = root
        self.mode = mode
        self.files_A = sorted(os.listdir(os.path.join(root, f"{mode}A")))
        self.files_B = sorted(os.listdir(os.path.join(root, f"{mode}B")))
        '''
        if max_samples:
            self.files_A = self.files_A[:max_samples]
            self.files_B = self.files_B[:max_samples]
        '''
        self.path_A = os.path.join(root, f"{mode}A")
        self.path_B = os.path.join(root, f"{mode}B")

    def __getitem__(self, index):
        img_A = Image.open(os.path.join(self.path_A, self.files_A[index % len(self.files_A)])).convert("RGB")
        img_B = Image.open(os.path.join(self.path_B, self.files_B[index % len(self.files_B)])).convert("RGB")

        if self.transform:
            img_A = self.transform(img_A)
            img_B = self.transform(img_B)

        return {"A": img_A, "B": img_B}

    def __len__(self):
        return max(len(self.files_A), len(self.files_B))


In [ ]:
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

### Generator

In [ ]:
# models.py
#  self attention helps the model learn long-range dependencies, especially useful for style transfer and texture consistency.
class SelfAttention(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.query = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.key   = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.value = nn.Conv2d(in_dim, in_dim, kernel_size=1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.size()
        proj_query = self.query(x).view(B, -1, H * W).permute(0, 2, 1)  # B x N x C'
        proj_key   = self.key(x).view(B, -1, H * W)                     # B x C' x N
        energy = torch.bmm(proj_query, proj_key)                       # B x N x N
        attention = torch.softmax(energy, dim=-1)                      # B x N x N
        proj_value = self.value(x).view(B, -1, H * W)                  # B x C x N

        out = torch.bmm(proj_value, attention.permute(0, 2, 1))        # B x C x N
        out = out.view(B, C, H, W)
        return self.gamma * out + x


class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Dropout(DROPOUT),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3),
            nn.InstanceNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)


class Generator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, num_residuals=6):
        super().__init__()
        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_channels, 64, kernel_size=7),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),
        ]

        # Downsampling
        in_features = 64
        out_features = in_features * 2
        for _ in range(2):  # 64 → 32 → 16
            model += [
                nn.Conv2d(in_features, out_features, kernel_size=3, stride=2, padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True),
            ]
            in_features = out_features
            out_features = in_features * 2

        # Residual blocks
        for _ in range(num_residuals):
            model += [ResidualBlock(in_features)]
        model += [SelfAttention(in_features)]

        # Upsampling
        out_features = in_features // 2
        for _ in range(2):  # 16 → 32 → 64
            model += [
                nn.ConvTranspose2d(in_features, out_features, kernel_size=3, stride=2, padding=1, output_padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True),
            ]
            in_features = out_features
            out_features = in_features // 2

        model += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(64, out_channels, kernel_size=7),
            nn.Tanh(),
        ]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)


### Discriminator

In [ ]:
# models.py (continued)
class Discriminator(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        def discriminator_block(in_filters, out_filters, normalization=True):
            layers = [nn.Conv2d(in_filters, out_filters, kernel_size=4, stride=2, padding=1)]
            if normalization:
                layers.append(nn.InstanceNorm2d(out_filters))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *discriminator_block(in_channels, 64, normalization=False),  # (64x64) -> (32x32)
            *discriminator_block(64, 128),  # -> (16x16)
            *discriminator_block(128, 256),  # -> (8x8)
            *discriminator_block(256, 512),  # -> (4x4)
            nn.Conv2d(512, 1, kernel_size=4, padding=1)  # Output patch size (3x3 or so)
        )

    def forward(self, x):
        return self.model(x)


### Loss functions

In [ ]:
adversarial_criterion = nn.MSELoss()
cycle_criterion = nn.L1Loss()
identity_criterion = nn.L1Loss()

In [ ]:
G_AB = Generator().to(device)
G_BA = Generator().to(device)
D_A = Discriminator().to(device)
D_B = Discriminator().to(device)

# Optimizers
g_optimizer = optim.Adam(
    itertools.chain(G_AB.parameters(), G_BA.parameters()), 
    lr=LR, betas=(0.5, 0.999)
)
d_A_optimizer = optim.Adam(D_A.parameters(), lr=LR, betas=(0.5, 0.999))
d_B_optimizer = optim.Adam(D_B.parameters(), lr=LR, betas=(0.5, 0.999))

### Training loop

In [ ]:
def train(dataloader):

    g_scheduler = torch.optim.lr_scheduler.StepLR(g_optimizer, step_size=1, gamma=0.97)
    d_A_scheduler = torch.optim.lr_scheduler.StepLR(d_A_optimizer, step_size=1, gamma=0.97)
    d_B_scheduler = torch.optim.lr_scheduler.StepLR(d_B_optimizer, step_size=1, gamma=0.97)

    for epoch in range(EPOCHS):
        for i, batch in enumerate(dataloader):
            real_A = batch["A"].to(device)
            real_B = batch["B"].to(device)

            # -----------------------------------
            #  Train Generators G_AB and G_BA
            # -----------------------------------
            g_optimizer.zero_grad()

            # Identity loss
            same_B = G_AB(real_B)
            loss_identity_B = identity_criterion(same_B, real_B) * LAMBDA_IDENTITY

            same_A = G_BA(real_A)
            loss_identity_A = identity_criterion(same_A, real_A) * LAMBDA_IDENTITY

            # GAN loss
            fake_B = G_AB(real_A)
            pred_fake_B = D_B(fake_B)
            loss_GAN_AB = adversarial_criterion(pred_fake_B, torch.ones_like(pred_fake_B))

            fake_A = G_BA(real_B)
            pred_fake_A = D_A(fake_A)
            loss_GAN_BA = adversarial_criterion(pred_fake_A, torch.ones_like(pred_fake_A))

            # Cycle loss
            recov_A = G_BA(fake_B)
            loss_cycle_A = cycle_criterion(recov_A, real_A)

            recov_B = G_AB(fake_A)
            loss_cycle_B = cycle_criterion(recov_B, real_B)

            # Total generator loss
            g_loss = (loss_GAN_AB + loss_GAN_BA) + \
                     LAMBDA_CYCLE * (loss_cycle_A + loss_cycle_B) + \
                     (loss_identity_A + loss_identity_B)

            g_loss.backward()
            g_optimizer.step()

            # -----------------------------------
            #  Train Discriminator A
            # -----------------------------------
            d_A_optimizer.zero_grad()

            pred_real = D_A(real_A)
            loss_D_real = adversarial_criterion(pred_real, torch.ones_like(pred_real))

            pred_fake = D_A(fake_A.detach())
            loss_D_fake = adversarial_criterion(pred_fake, torch.zeros_like(pred_fake))

            d_A_loss = (loss_D_real + loss_D_fake) * 0.5
            d_A_loss.backward()
            d_A_optimizer.step()

            # -----------------------------------
            #  Train Discriminator B
            # -----------------------------------
            d_B_optimizer.zero_grad()

            pred_real = D_B(real_B)
            loss_D_real = adversarial_criterion(pred_real, torch.ones_like(pred_real))

            pred_fake = D_B(fake_B.detach())
            loss_D_fake = adversarial_criterion(pred_fake, torch.zeros_like(pred_fake))

            d_B_loss = (loss_D_real + loss_D_fake) * 0.5
            d_B_loss.backward()
            d_B_optimizer.step()

            torch.cuda.empty_cache()
            gc.collect()


        g_scheduler.step()
        d_A_scheduler.step()
        d_B_scheduler.step()


        print(
                    f"[Epoch {epoch}/{EPOCHS}] [Batch {i}/{len(dataloader)}] "
                    f"[D_A Loss: {d_A_loss.item():.4f}] [D_B Loss: {d_B_loss.item():.4f}] "
                    f"[G Loss: {g_loss.item():.4f}]"
                )
    torch.save(G_AB.state_dict(), 'G_AB_model.pth')
    torch.save(G_BA.state_dict(), 'G_BA_model.pth')
    torch.save(D_A.state_dict(), 'D_A_model.pth')
    torch.save(D_B.state_dict(), 'D_B_model.pth')

In [ ]:
accumulation_steps = 8  # Tune this based on your memory capacity

def train_accumulation(dataloader):
    for epoch in range(EPOCHS):
        g_optimizer.zero_grad()
        d_A_optimizer.zero_grad()
        d_B_optimizer.zero_grad()

        # Initialize accumulators
        total_g_loss = 0.0
        total_d_A_loss = 0.0
        total_d_B_loss = 0.0
        num_batches = 0

        for i, batch in enumerate(dataloader):
            real_A = batch["A"].to(device)
            real_B = batch["B"].to(device)

            # ========== Train Generators ==========
            same_B = G_AB(real_B)
            loss_identity_B = identity_criterion(same_B, real_B) * LAMBDA_IDENTITY

            same_A = G_BA(real_A)
            loss_identity_A = identity_criterion(same_A, real_A) * LAMBDA_IDENTITY

            fake_B = G_AB(real_A)
            pred_fake_B = D_B(fake_B)
            loss_GAN_AB = adversarial_criterion(pred_fake_B, torch.ones_like(pred_fake_B))

            fake_A = G_BA(real_B)
            pred_fake_A = D_A(fake_A)
            loss_GAN_BA = adversarial_criterion(pred_fake_A, torch.ones_like(pred_fake_A))

            recov_A = G_BA(fake_B)
            loss_cycle_A = cycle_criterion(recov_A, real_A)

            recov_B = G_AB(fake_A)
            loss_cycle_B = cycle_criterion(recov_B, real_B)

            g_loss = (loss_GAN_AB + loss_GAN_BA) + \
                     LAMBDA_CYCLE * (loss_cycle_A + loss_cycle_B) + \
                     (loss_identity_A + loss_identity_B)

            total_g_loss += g_loss.item()
            g_loss = g_loss / accumulation_steps
            g_loss.backward()



            # ========== Train Discriminator A ==========
            pred_real = D_A(real_A)
            loss_D_real = adversarial_criterion(pred_real, torch.ones_like(pred_real))

            pred_fake = D_A(fake_A.detach())
            loss_D_fake = adversarial_criterion(pred_fake, torch.zeros_like(pred_fake))

            d_A_loss = (loss_D_real + loss_D_fake) * 0.5
            total_d_A_loss += d_A_loss.item()
            d_A_loss = d_A_loss / accumulation_steps
            d_A_loss.backward()



            # ========== Train Discriminator B ==========
            pred_real = D_B(real_B)
            loss_D_real = adversarial_criterion(pred_real, torch.ones_like(pred_real))

            pred_fake = D_B(fake_B.detach())
            loss_D_fake = adversarial_criterion(pred_fake, torch.zeros_like(pred_fake))

            d_B_loss = (loss_D_real + loss_D_fake) * 0.5
            total_d_B_loss += d_B_loss.item()
            d_B_loss = d_B_loss / accumulation_steps
            d_B_loss.backward()


            if (num_batches + 1) % accumulation_steps == 0:
                g_optimizer.step()
                g_optimizer.zero_grad()
                d_A_optimizer.step()
                d_A_optimizer.zero_grad()
                d_B_optimizer.step()
                d_B_optimizer.zero_grad()



            num_batches += 1

            torch.cuda.empty_cache()
            gc.collect()


        # Final step for leftover gradients
        if (num_batches + 1) % accumulation_steps != 0:
            g_optimizer.step()
            d_A_optimizer.step()
            d_B_optimizer.step()
            g_optimizer.zero_grad()
            d_A_optimizer.zero_grad()
            d_B_optimizer.zero_grad()

        # Print average losses
        print(
            f"[Epoch {epoch}/{EPOCHS}] "
            f"[Avg D_A Loss: {total_d_A_loss / num_batches:.4f}] "
            f"[Avg D_B Loss: {total_d_B_loss / num_batches:.4f}] "
            f"[Avg G Loss: {total_g_loss / num_batches:.4f}]"
        )

        print(
                    f"[Epoch {epoch}/{EPOCHS}] [Batch {i}/{len(dataloader)}] "
                    f"[D_A Loss: {d_A_loss.item():.4f}] [D_B Loss: {d_B_loss.item():.4f}] "
                    f"[G Loss: {g_loss.item():.4f}]"
                )
    torch.save(G_AB.state_dict(), 'G_AB_model_acc2.pth')
    torch.save(G_BA.state_dict(), 'G_BA_model_acc2.pth')
    torch.save(D_A.state_dict(), 'D_A_model_acc2.pth')
    torch.save(D_B.state_dict(), 'D_B_model_acc2.pth')

In [ ]:
if __name__ == "__main__":
    dataset = ImageDataset(path_to_dataset, transform=transform)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, )
    train(dataloader)

In [ ]:
def evaluate_cycle_consistency(G_AB, G_BA, dataloader, device):
    G_AB.eval()
    G_BA.eval()

    total_loss_A = 0.0
    total_loss_B = 0.0
    num_samples = 0

    with torch.no_grad():
        for batch in dataloader:
            real_A = batch["A"].to(device)
            real_B = batch["B"].to(device)

            # A → B → A
            fake_B = G_AB(real_A)
            recon_A = G_BA(fake_B)
            loss_A = cycle_criterion(recon_A, real_A)

            # B → A → B
            fake_A = G_BA(real_B)
            recon_B = G_AB(fake_A)
            loss_B = cycle_criterion(recon_B, real_B)

            total_loss_A += loss_A.item()
            total_loss_B += loss_B.item()
            num_samples += real_A.size(0)  # batch size (should be 1 for test)

    avg_loss_A = total_loss_A / num_samples
    avg_loss_B = total_loss_B / num_samples

    print(f"Cycle Consistency Loss A→B→A: {avg_loss_A:.4f}")
    print(f"Cycle Consistency Loss B→A→B: {avg_loss_B:.4f}")
    print(f"Average Cycle Consistency Loss: {(avg_loss_A + avg_loss_B) / 2:.4f}")

    return avg_loss_A, avg_loss_B

test_dataset = ImageDataset(path_to_dataset, transform=transform, mode='test')
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

evaluate_cycle_consistency(G_AB, G_BA, test_loader, device)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the models and move them to the selected device
G_AB = Generator().to(device)
G_BA = Generator().to(device)
D_A = Discriminator().to(device)
D_B = Discriminator().to(device) # Your actual Discriminator class

G_AB.load_state_dict(torch.load('G_AB_model.pth'))
G_BA.load_state_dict(torch.load('G_BA_model.pth'))
D_A.load_state_dict(torch.load('D_A_model.pth'))
D_B.load_state_dict(torch.load('D_B_model.pth'))

#attempt 50 epoch a accumulatimg gradient nedopadlo dobre

# Set models to evaluation mode
G_AB.eval()
G_BA.eval()
D_A.eval()
D_B.eval()

test_dataset = ImageDataset(path_to_dataset, transform=transform, mode='test', max_samples=50
                           )
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)


In [ ]:
test_dataset = ImageDataset(path_to_dataset, transform=transform, mode='test')
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [ ]:
def show_images(real, fake, title="Input vs Output"):
    real = real * 0.5 + 0.5
    fake = fake * 0.5 + 0.5

    images = torch.cat([real, fake], dim=3)
    grid = make_grid(images, nrow=1)

    np_img = grid.permute(1, 2, 0).detach().cpu().numpy()
    plt.figure(figsize=(6, 3))
    plt.imshow(np_img)
    plt.axis("off")
    plt.title(title)
    plt.show()



In [ ]:
# cycle gan static learning
G_AB.eval()

with torch.no_grad():
    for i, batch in enumerate(test_loader):
        real_A = batch["A"].to(device)
        fake_B = G_AB(real_A)
        show_images(real_A, fake_B, title=f"Sample {i+1}")

In [ ]:
# Cyclegan 50 epoch dynamic leraning
G_AB.eval()

with torch.no_grad():
    for i, batch in enumerate(test_loader):
        real_A = batch["A"].to(device)
        fake_B = G_AB(real_A)
        show_images(real_A, fake_B, title=f"Sample {i+1}")

## AMS cycleGAN

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
IMG_SIZE = 128
EPOCHS = 50
LAMBDA_CYCLE = 10
LAMBDA_IDENTITY = 5
LAMBDA_STYLE = 10
LEARNING_RATE = 0.0002
DROPOUT = 0.5
path_to_dataset="data/horse2zebra/horse2zebra"

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [-1, 1]
])

### Dataset

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, root, transform=None, mode="train", max_samples=None):
        self.transform = transform
        self.files_A = sorted(os.listdir(os.path.join(root, f"{mode}A")))
        self.files_B = sorted(os.listdir(os.path.join(root, f"{mode}B")))

        if max_samples:
            self.files_A = self.files_A[:max_samples]
            self.files_B = self.files_B[:max_samples]

        self.path_A = os.path.join(root, f"{mode}A")
        self.path_B = os.path.join(root, f"{mode}B")

    def __getitem__(self, index):
        img_A = Image.open(os.path.join(self.path_A, self.files_A[index % len(self.files_A)])).convert("RGB")
        img_B = Image.open(os.path.join(self.path_B, self.files_B[index % len(self.files_B)])).convert("RGB")

        if self.transform:
            img_A = self.transform(img_A)
            img_B = self.transform(img_B)

        return {"A": img_A, "B": img_B}

    def __len__(self):
        return max(len(self.files_A), len(self.files_B))


In [ ]:
class ResnetBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3),
            nn.InstanceNorm2d(dim),
            nn.ReLU(inplace=True),
            nn.Dropout(DROPOUT),
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3),
            nn.InstanceNorm2d(dim)
        )

    def forward(self, x):
        return x + self.block(x)


In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.ReflectionPad2d(3),
            nn.Conv2d(3, 64, 7),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.ReLU(inplace=True),

            ResnetBlock(256),
            ResnetBlock(256),
            ResnetBlock(256),
            ResnetBlock(256),
            ResnetBlock(256),

            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),

            nn.ReflectionPad2d(3),
            nn.Conv2d(64, 3, 7),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2),

            nn.Conv2d(256, 512, 4, padding=1),
            nn.InstanceNorm2d(512),
            nn.LeakyReLU(0.2),

            nn.Conv2d(512, 1, 4, padding=1)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
def gram_matrix(tensor):
    b, c, h, w = tensor.size()
    features = tensor.view(b, c, h * w)
    gram = torch.bmm(features, features.transpose(1, 2))
    return gram / (c * h * w)

def style_loss(fake, real):
    return F.l1_loss(gram_matrix(fake), gram_matrix(real))

# Display Function
def show_images(real, fake, title="Input vs Output"):
    real = real * 0.5 + 0.5
    fake = fake * 0.5 + 0.5
    images = torch.cat([real, fake], dim=3)
    grid = make_grid(images, nrow=1)
    np_img = grid.permute(1, 2, 0).detach().cpu().numpy()
    plt.figure(figsize=(6, 3))
    plt.imshow(np_img)
    plt.axis("off")
    plt.title(title)
    plt.show()

In [ ]:
G_AB = Generator().to(DEVICE)
G_BA = Generator().to(DEVICE)
D_A = Discriminator().to(DEVICE)
D_B = Discriminator().to(DEVICE)

# Loss functions
adversarial_criterion = nn.MSELoss()
cycle_criterion = nn.L1Loss()
identity_criterion = nn.L1Loss()

# Optimizers
g_optimizer = torch.optim.Adam(
    list(G_AB.parameters()) + list(G_BA.parameters()), lr=LEARNING_RATE, betas=(0.5, 0.999)
)
d_optimizer_A = torch.optim.Adam(D_A.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))
d_optimizer_B = torch.optim.Adam(D_B.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))

# Load Dataset
dataset = ImageDataset(path_to_dataset, transform=transform, mode="train")
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
g_scheduler = StepLR(g_optimizer, step_size=1, gamma=0.97)
d_scheduler_A = StepLR(d_optimizer_A, step_size=1, gamma=0.97)
d_scheduler_B = StepLR(d_optimizer_B, step_size=1, gamma=0.97)

for epoch in range(EPOCHS):
    for i, batch in enumerate(dataloader):
        real_A = batch["A"].to(DEVICE)
        real_B = batch["B"].to(DEVICE)

        # Train Generators
        g_optimizer.zero_grad()

        same_B = G_AB(real_B)
        loss_identity_B = identity_criterion(same_B, real_B) * LAMBDA_IDENTITY

        same_A = G_BA(real_A)
        loss_identity_A = identity_criterion(same_A, real_A) * LAMBDA_IDENTITY

        fake_B = G_AB(real_A)
        pred_fake_B = D_B(fake_B)
        loss_GAN_AB = adversarial_criterion(pred_fake_B, torch.ones_like(pred_fake_B))

        fake_A = G_BA(real_B)
        pred_fake_A = D_A(fake_A)
        loss_GAN_BA = adversarial_criterion(pred_fake_A, torch.ones_like(pred_fake_A))

        recov_A = G_BA(fake_B)
        loss_cycle_A = cycle_criterion(recov_A, real_A)

        recov_B = G_AB(fake_A)
        loss_cycle_B = cycle_criterion(recov_B, real_B)

        loss_style_B = style_loss(fake_B, real_B)
        loss_style_A = style_loss(fake_A, real_A)

        g_loss = (loss_GAN_AB + loss_GAN_BA) + \
                 LAMBDA_CYCLE * (loss_cycle_A + loss_cycle_B) + \
                 (loss_identity_A + loss_identity_B) + \
                 LAMBDA_STYLE * (loss_style_A + loss_style_B)

        g_loss.backward()
        g_optimizer.step()

        # Train Discriminator A
        d_optimizer_A.zero_grad()
        pred_real = D_A(real_A)
        pred_fake = D_A(fake_A.detach())
        d_loss_A = (adversarial_criterion(pred_real, torch.ones_like(pred_real)) +
                    adversarial_criterion(pred_fake, torch.zeros_like(pred_fake))) * 0.5
        d_loss_A.backward()
        d_optimizer_A.step()

        # Train Discriminator B
        d_optimizer_B.zero_grad()
        pred_real = D_B(real_B)
        pred_fake = D_B(fake_B.detach())
        d_loss_B = (adversarial_criterion(pred_real, torch.ones_like(pred_real)) +
                    adversarial_criterion(pred_fake, torch.zeros_like(pred_fake))) * 0.5
        d_loss_B.backward()
        d_optimizer_B.step()

        torch.cuda.empty_cache()
        gc.collect()


    print(
        f"[Epoch {epoch}/{EPOCHS}] [Batch {i}/{len(dataloader)}] "
        f"[D_A Loss: {d_loss_A.item():.4f}] [D_B Loss: {d_loss_B.item():.4f}] "
        f"[G Loss: {g_loss.item():.4f}]"
        )

    g_scheduler.step()
    d_scheduler_A.step()
    d_scheduler_B.step()

In [ ]:
torch.save(G_AB.state_dict(), 'G_AB_model_AMS.pth')
torch.save(G_BA.state_dict(), 'G_BA_model_AMS.pth')
torch.save(D_A.state_dict(), 'D_A_model_AMS.pth')
torch.save(D_B.state_dict(), 'D_B_model_AMS.pth')

In [ ]:
test_dataset = ImageDataset(path_to_dataset, transform=transform, mode="test" , max_samples=20 )
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

G_AB.eval()
with torch.no_grad():
    for i, batch in enumerate(test_loader):
        real_A = batch["A"].to(DEVICE)
        fake_B = G_AB(real_A)
        show_images(real_A, fake_B, title=f"AMS-CycleGAN Output {i+1}")

In [ ]:
def show_triplet(real_A, fake_B, recon_A, title="A → B → A"):
    real_A = real_A * 0.5 + 0.5
    fake_B = fake_B * 0.5 + 0.5
    recon_A = recon_A * 0.5 + 0.5

    images = torch.cat([real_A, fake_B, recon_A], dim=3)
    grid = make_grid(images, nrow=1)
    np_img = grid.permute(1, 2, 0).cpu().numpy()

    plt.figure(figsize=(9, 3))
    plt.imshow(np_img)
    plt.axis("off")
    plt.title(title)
    plt.show()


def visualize_cycle_consistency(G_AB, G_BA, dataloader, device, num_samples=5):
    G_AB.eval()
    G_BA.eval()

    with torch.no_grad():
        for i, batch in enumerate(dataloader):
            if i >= num_samples:
                break

            real_A = batch["A"].to(device)
            fake_B = G_AB(real_A)
            recon_A = G_BA(fake_B)

            show_triplet(real_A, fake_B, recon_A, title=f"A → B → A (Sample {i+1})")


visualize_cycle_consistency(G_AB, G_BA, test_loader, device, num_samples=30)